In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.ticker as ticker
from viz_config import VizConfig

# ==========================================
# 0. Global configuration and style setup
# ==========================================
# Apply the shared colour/style configuration (viz_config.py)
VizConfig.set_style()

# Pull font-size constants from VizConfig
TITLE_SIZE = VizConfig.TITLE_SIZE
LABEL_SIZE = VizConfig.LABEL_SIZE
TICK_SIZE = VizConfig.TICK_SIZE
LEGEND_SIZE = VizConfig.LEGEND_SIZE

# Path configuration
DATA_SUBDIR = "Data_Efficiency_Results"
CSV_NAME = "Data_Efficiency_Curve.csv"
FULL_PATH = os.path.join(DATA_SUBDIR, CSV_NAME)

# Make sure the output directory exists
if not os.path.exists(DATA_SUBDIR):
    os.makedirs(DATA_SUBDIR)

# ==========================================
# 1. Data loading
# ==========================================
print(f"Reading data: {FULL_PATH}")

if not os.path.exists(FULL_PATH):
    print(f"Warning: {FULL_PATH} not found. Using dummy data.")
    # Build sample data (if the real file is missing)
    # Training_Size: Training-set size
    # Mean_R2: Test-set R^2 mean
    # Std_R2: Test-set R^2 std (for error bars)
    data = {'Training_Size': [100, 500, 1000, 2000, 4000, 10000, 20000],
            'Mean_R2': [0.6, 0.8, 0.9, 0.95, 0.97, 0.98, 0.985],
            'Std_R2': [0.1, 0.08, 0.05, 0.03, 0.02, 0.01, 0.01]}
    df = pd.DataFrame(data)
else:
    df = pd.read_csv(FULL_PATH)

# Sort by training-set size so the connecting line is correct
df = df.sort_values('Training_Size')
x = df['Training_Size'].values
y = df['Mean_R2'].values
yerr = df['Std_R2'].values

# ==========================================
# 2. Data analysis and key-point computation
# ==========================================
max_r2 = np.max(y)
max_idx = np.argmax(y)

# Pick a threshold to locate"optimal efficiency point" (Optimal Point)
# Defined as the smallest sample count reaching 99% of the peak R^2
threshold = max_r2 * 0.99
optimal_indices = np.where(y >= threshold)[0]
# Select the first point above the threshold (smallest sample count)
best_idx = optimal_indices[0] if len(optimal_indices) > 0 else max_idx
best_size = x[best_idx]
best_r2 = y[best_idx]

# Define"high-efficiency zone" (High Efficiency Zone)  X range
# Range from just-before-optimal to just-after-max (or proportionally)
zone_min = x[max(0, best_idx - 1)] if best_idx > 0 else best_size * 0.5
zone_max = x[min(len(x)-1, max_idx + 1)] if max_idx < len(x)-1 else x[max_idx] * 1.5

# ==========================================
# 3. Plotting (Figure 8: Data Efficiency)
# ==========================================
fig, ax = plt.subplots(figsize=(10, 7))

# 3.1 Colour definitions (from VizConfig)
COLOR_MAIN = VizConfig.COLOR_MAIN       # Main-curve colour (deep blue)
COLOR_ERROR = VizConfig.COLOR_SECONDARY # Error-bar colour (grey)
COLOR_OPTIMAL = VizConfig.COLOR_HIGHLIGHT # Optimal-point colour (red)
COLOR_HATCH = VizConfig.COLOR_SUCCESS   # High-efficiency zone colour (green)

# 3.2 Plot the high-efficiency zone
# Use a hatched (hatch='//')  rectangle to mark it
ax.axvspan(zone_min, zone_max, 
           facecolor='none',       # transparent background
           hatch='//',             # Hatch fill
           edgecolor=COLOR_HATCH,  # Hatch colour
           alpha=0.3,              # alpha (transparency)
           zorder=0)               # Send to the back

# Compute the Y axis range
y_min_val = min(y - yerr)
y_lower = max(0, y_min_val - 0.1)
ax.set_ylim(y_lower, 1.02)

# Add the zone annotation "High Efficiency Zone"
# Log scale geometric center: Geometric centre on the log axis
zone_center = np.sqrt(zone_min * zone_max) 
# transform=ax.get_xaxis_transform(): x Data coords for X, axis-fraction coords for Y (0-1)
ax.text(zone_center, 0.05, "High Efficiency Zone", 
        transform=ax.get_xaxis_transform(), # y=0.05 5% above the bottom
        ha='center', va='bottom', fontsize=12, fontweight='bold', 
        color=COLOR_HATCH, style='italic', 
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=3))

# 3.3 Plot the error-bar curve
# Semi-log axes (log on X)
ax.errorbar(x, y, yerr=yerr, fmt='-o', markersize=8, linewidth=2.5, capsize=4,
             color=COLOR_MAIN, ecolor=COLOR_ERROR, elinewidth=1.5, 
             markerfacecolor='white', markeredgecolor=COLOR_MAIN, 
             markeredgewidth=2, label='Mean $R^2$ Score', zorder=3)

# 3.4 Mark the optimal efficiency point
ax.scatter(best_size, best_r2, s=120, color=COLOR_OPTIMAL, alpha=1.0, 
          edgecolor='white', linewidth=2, zorder=4, label='Optimal Point')

# Vertical dashed line through the optimal point
ax.axvline(x=best_size, color=COLOR_OPTIMAL, linestyle='--', linewidth=1.5, alpha=0.6, ymax=0.95, zorder=2)

# 3.5 Annotation text
# Auto-adjust annotation position to avoid covering the curve
xy_text_pos = (best_size * 2, best_r2 - 0.08) if best_r2 > 0.8 else (best_size * 2, best_r2 + 0.1)
ax.annotate(f'Optimal Sample Size\nN = {int(best_size)}', 
            xy=(best_size, best_r2), xytext=xy_text_pos,
            arrowprops=dict(arrowstyle="->", color=VizConfig.COLOR_AXIS, lw=1.5, connectionstyle="arc3,rad=-0.2"),
            fontsize=12, color=VizConfig.COLOR_AXIS, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#cfd8dc", alpha=0.95))

# ==========================================
# 4. Axes and spines styling
# ==========================================
# Set X axis to log scale
ax.set_xscale('log')

# Axis labels and title
ax.set_xlabel("Training Dataset Size (Log Scale)", fontsize=LABEL_SIZE, fontweight='bold', labelpad=10)
ax.set_ylabel(r"Test Set $R^2$ Score", fontsize=LABEL_SIZE, fontweight='bold', labelpad=10)
ax.set_title("Data Efficiency Analysis", fontsize=TITLE_SIZE, pad=20, fontweight='bold')

# Grid configuration
ax.grid(True, which="major", linestyle='-', alpha=0.6, color=VizConfig.COLOR_GRID, linewidth=1)
ax.grid(True, which="minor", linestyle=':', alpha=0.4, color=VizConfig.COLOR_GRID, linewidth=0.8)

# Spine styling (bold every spine)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2.0)
    spine.set_color(VizConfig.COLOR_AXIS)

# Tick styling (inward ticks)
ax.tick_params(axis='both', which='major', labelsize=TICK_SIZE, length=6, width=1.5, colors=VizConfig.COLOR_AXIS, direction='in')
ax.tick_params(axis='both', which='minor', length=3, width=1, colors=VizConfig.COLOR_AXIS, direction='in')

# Legend configuration
legend = ax.legend(fontsize=LEGEND_SIZE, loc='lower right', frameon=True, edgecolor=VizConfig.COLOR_AXIS, fancybox=False)
legend.get_frame().set_linewidth(1.5)

# ==========================================
# 5. Save output
# ==========================================
plt.tight_layout()
output_path = os.path.join(DATA_SUBDIR, "8.pdf")
# Save as a high-resolution PDF
plt.savefig(output_path, dpi=VizConfig.DPI, bbox_inches='tight', pad_inches=0.1) 

print(f"Saved refined plot to {output_path}")
plt.show()